# 2. Asteroid Risk Prediction System - Data Preprocessing

**Copyright © 2026 StellarMind - EarthGuard Asteroid Defense AI**  
**All Rights Reserved.**

**Authors**: Gouragopal Mohapatra & Arijit Kumar Mohanty  
**Project**: StellarMind - EarthGuard Asteroid Defense AI  
**Goal**: Clean, scale, and prepare asteroid data for RF, DT, and Regression models

**License**: This code and its contents are proprietary to StellarMind.  
Unauthorized copying, distribution, or use of this material is prohibited without prior written permission.

**Version**: 1.0 | **Date**: April 2026

In [36]:
# 2. Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

In [37]:
# 3. Load cleaned data from notebook 01
df = pd.read_csv('../data/processed/asteroid_cleaned.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
df.head(2)

Dataset shape: (25869, 44)
Columns: 44


,Unnamed: 0,id,orbit_id,orbit_determination_date,first_observation_date,last_observation_date,data_arc_in_days,observations_used,orbit_uncertainty,minimum_orbit_intersection,...,relative_velocity0,orbiting_body1,miss_distance_kilometers1,relative_velocity1,orbiting_body2,miss_distance_kilometers2,relative_velocity2,orbiting_body3,miss_distance_kilometers3,relative_velocity3
0,0,2000433,659,2021-05-24 17:55:05,1893-10-29,2021-05-13,46582.0,9130,0,0.148353,...,5.578619,Earth,7.053323e+07,4.394491,Earth,7.468781e+07,4.816784,Earth,5.382329e+07,4.596055
1,1,2000719,272,2025-03-28 06:20:45,1911-10-04,2025-03-28,41449.0,2102,0,0.201318,...,3.446029,Juptr,2.119996e+08,3.263461,Juptr,2.223920e+08,3.724128,Juptr,2.930859e+08,5.004843


In [38]:
# 4. Define target variable and features
target_col = 'is_potentially_hazardous_asteroid'
print(f"Target column: {target_col}")
print(f"Target distribution:\n{df[target_col].value_counts()}")

# Separate features and target
X = df.drop(columns=[target_col])
y = df[target_col]
print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")

Target column: is_potentially_hazardous_asteroid
Target distribution:
is_potentially_hazardous_asteroid
False    23370
True      2499
Name: count, dtype: int64

Features shape: (25869, 43)
Target shape: (25869,)


In [39]:
# 5. Identify numeric vs categorical columns
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric columns: {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")
print(f"\nCategorical columns sample: {categorical_cols[:5]}")

Numeric columns: 32
Categorical columns: 11

Categorical columns sample: ['orbit_determination_date', 'first_observation_date', 'last_observation_date', 'equinox', 'orbit_class_type']


In [40]:
# 6. Handle missing values in numeric columns
print("=== Missing Values Before Imputation ===")
missing_before = X[numeric_cols].isnull().sum()
missing_before = missing_before[missing_before > 0]
print(f"Columns with missing values: {len(missing_before)}")
print(missing_before.head(10))

# Impute numeric columns with median
imputer = SimpleImputer(strategy='median')
X[numeric_cols] = imputer.fit_transform(X[numeric_cols])

print(f"\n✓ Missing values imputed with median strategy")

=== Missing Values Before Imputation ===
Columns with missing values: 1
data_arc_in_days    137
dtype: int64

✓ Missing values imputed with median strategy


In [41]:
# 7. Handle missing values in categorical columns (if any)
if categorical_cols:
    print("=== Categorical Columns Missing Values ===")
    cat_missing = X[categorical_cols].isnull().sum()
    cat_missing = cat_missing[cat_missing > 0]
    print(f"Columns with missing values: {len(cat_missing)}")
    
    if len(cat_missing) > 0:
        # Impute categorical with most frequent
        cat_imputer = SimpleImputer(strategy='most_frequent')
        X[categorical_cols] = cat_imputer.fit_transform(X[categorical_cols])
        print("✓ Categorical missing values imputed with most_frequent strategy")
    else:
        print("✓ No missing values in categorical columns")
else:
    print("✓ No categorical columns to process")

=== Categorical Columns Missing Values ===
Columns with missing values: 0
✓ No missing values in categorical columns


In [42]:
# 8. Remove unnecessary identifier columns
id_cols = ['Unnamed: 0', 'id', 'neo_reference_id', 'orbit_id', 'name']
cols_to_drop = [col for col in id_cols if col in X.columns]

if cols_to_drop:
    X = X.drop(columns=cols_to_drop)
    print(f"✓ Removed identifier columns: {cols_to_drop}")
else:
    print("✓ No identifier columns found")
    
print(f"Features shape after dropping IDs: {X.shape}")

✓ Removed identifier columns: ['Unnamed: 0', 'id', 'neo_reference_id', 'orbit_id', 'name']
Features shape after dropping IDs: (25869, 38)


In [43]:
# 9. Check for infinite values and handle them
print("=== Checking for Infinite Values ===")
numeric_cols_updated = X.select_dtypes(include=[np.number]).columns.tolist()
infinite_check = np.isinf(X[numeric_cols_updated]).sum().sum()
print(f"Infinite values found: {infinite_check}")

if infinite_check > 0:
    X[numeric_cols_updated] = X[numeric_cols_updated].replace([np.inf, -np.inf], np.nan)
    X[numeric_cols_updated] = imputer.fit_transform(X[numeric_cols_updated])
    print("✓ Infinite values replaced with NaN and imputed")

=== Checking for Infinite Values ===
Infinite values found: 0


In [44]:
# 10. Compare different scaling methods
print("=== Scaling Methods Comparison ===")
print("1. StandardScaler (z-score normalization)")
print("2. MinMaxScaler (range 0-1)")
print("3. RobustScaler (robust to outliers)")

# We'll use RobustScaler due to outliers detected in EDA
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X[numeric_cols_updated])
X_scaled = pd.DataFrame(X_scaled, columns=numeric_cols_updated)

print(f"\n✓ Applied RobustScaler to {len(numeric_cols_updated)} features")
print(f"Scaled data shape: {X_scaled.shape}")

=== Scaling Methods Comparison ===
1. StandardScaler (z-score normalization)
2. MinMaxScaler (range 0-1)
3. RobustScaler (robust to outliers)

✓ Applied RobustScaler to 28 features
Scaled data shape: (25869, 28)


In [45]:
# 11. Verify scaling worked correctly
print("=== Before Scaling (Sample Feature) ===")
sample_feature = numeric_cols_updated[0]
print(f"{sample_feature}: mean={X[sample_feature].mean():.4f}, std={X[sample_feature].std():.4f}")

print("\n=== After Scaling (Sample Feature) ===")
print(f"{sample_feature}: mean={X_scaled[sample_feature].mean():.4f}, std={X_scaled[sample_feature].std():.4f}")

=== Before Scaling (Sample Feature) ===
data_arc_in_days: mean=2005.6730, std=3420.6895

=== After Scaling (Sample Feature) ===
data_arc_in_days: mean=0.5910, std=1.0407


In [46]:
# 12. Handle categorical features with encoding (NO EXPLOSION)
if categorical_cols and len(categorical_cols) > 0:
    print("=== Encoding Categorical Features ===")
    print(f"Categorical columns to encode: {categorical_cols}")
    
    categorical_cols_existing = [col for col in categorical_cols if col in X.columns]
    
    if categorical_cols_existing:
        X_final = X_scaled.copy()
        
        for col in categorical_cols_existing:
            unique_count = X[col].nunique()
            print(f"  {col}: {unique_count} unique values")
            
            if unique_count <= 10:
                # Low cardinality - use one-hot encoding
                dummies = pd.get_dummies(X[col], prefix=col, drop_first=True)
                X_final = pd.concat([X_final.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
                print(f"    ✓ One-hot encoded -> {dummies.shape[1]} new columns")
                
            elif unique_count <= 50:
                # Medium cardinality - use frequency encoding
                freq_map = X[col].value_counts(normalize=True).to_dict()
                X_final[col + '_freq'] = X[col].map(freq_map)
                print(f"    ✓ Frequency encoded -> 1 new column")
                
            else:
                # High cardinality - extract only top 5 categories
                top_categories = X[col].value_counts().head(5).index
                for cat in top_categories:
                    X_final[col + '_' + str(cat)] = (X[col] == cat).astype(int)
                print(f"    ✓ Top-5 encoded -> 5 new columns (avoided {unique_count} columns)")
        
        # Drop original categorical columns
        X_final = X_final.drop(columns=categorical_cols_existing, errors='ignore')
        print(f"\n✓ Final feature shape: {X_final.shape}")
        
    else:
        X_final = X_scaled
        print("✓ No categorical columns found after cleaning")
else:
    X_final = X_scaled
    print("✓ No categorical columns to encode")
    print(f"Final feature shape: {X_final.shape}")

=== Encoding Categorical Features ===
Categorical columns to encode: ['orbit_determination_date', 'first_observation_date', 'last_observation_date', 'equinox', 'orbit_class_type', 'orbit_class_description', 'name', 'orbiting_body0', 'orbiting_body1', 'orbiting_body2', 'orbiting_body3']
  orbit_determination_date: 15269 unique values
    ✓ Top-5 encoded -> 5 new columns (avoided 15269 columns)
  first_observation_date: 7001 unique values
    ✓ Top-5 encoded -> 5 new columns (avoided 7001 columns)
  last_observation_date: 5061 unique values
    ✓ Top-5 encoded -> 5 new columns (avoided 5061 columns)
  equinox: 1 unique values
    ✓ One-hot encoded -> 0 new columns
  orbit_class_type: 4 unique values
    ✓ One-hot encoded -> 3 new columns
  orbit_class_description: 4 unique values
    ✓ One-hot encoded -> 3 new columns
  orbiting_body0: 8 unique values
    ✓ One-hot encoded -> 7 new columns
  orbiting_body1: 7 unique values
    ✓ One-hot encoded -> 6 new columns
  orbiting_body2: 7 unique

In [47]:
# 13. Train-test split for model evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42, stratify=y
)

print("=== Train-Test Split Results ===")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"\nTrain target distribution:\n{y_train.value_counts(normalize=True)}")
print(f"\nTest target distribution:\n{y_test.value_counts(normalize=True)}")

=== Train-Test Split Results ===
X_train shape: (20695, 73)
X_test shape: (5174, 73)
y_train shape: (20695,)
y_test shape: (5174,)

Train target distribution:
is_potentially_hazardous_asteroid
False    0.903407
True     0.096593
Name: proportion, dtype: float64

Test target distribution:
is_potentially_hazardous_asteroid
False    0.903363
True     0.096637
Name: proportion, dtype: float64


In [48]:
# 14. Check for class imbalance
print("=== Class Imbalance Check ===")
hazardous_pct_train = y_train.mean() * 100
hazardous_pct_test = y_test.mean() * 100

print(f"Training set - Hazardous: {hazardous_pct_train:.2f}%")
print(f"Test set - Hazardous: {hazardous_pct_test:.2f}%")

if hazardous_pct_train < 10 or hazardous_pct_train > 90:
    print("⚠️ Warning: Significant class imbalance detected")
    print("Consider using class_weight='balanced' in models")
else:
    print("✓ Class distribution is reasonably balanced")

=== Class Imbalance Check ===
Training set - Hazardous: 9.66%
Test set - Hazardous: 9.66%
⚠️ Warning: Significant class imbalance detected
Consider using class_weight='balanced' in models


In [49]:
# 15. Save preprocessed data for modeling
import time

# Save train and test sets
train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

print("=== Saving Files ===")

# Save train.csv
start_train = time.time()
train_df.to_csv('../data/processed/train.csv', index=False)
train_time = time.time() - start_train
print(f"✓ Train set saved to '../data/processed/train.csv'")
print(f"  Shape: {train_df.shape} | Time: {train_time:.2f} seconds")

# Save test.csv
start_test = time.time()
test_df.to_csv('../data/processed/test.csv', index=False)
test_time = time.time() - start_test
print(f"✓ Test set saved to '../data/processed/test.csv'")
print(f"  Shape: {test_df.shape} | Time: {test_time:.2f} seconds")

print(f"\n✓ Total save time: {train_time + test_time:.2f} seconds")

=== Saving Files ===
✓ Train set saved to '../data/processed/train.csv'
  Shape: (20695, 74) | Time: 0.56 seconds
✓ Test set saved to '../data/processed/test.csv'
  Shape: (5174, 74) | Time: 0.14 seconds

✓ Total save time: 0.70 seconds


In [50]:
# Check what's causing the slowdown
print("=== DIAGNOSTIC ===")
print(f"1. X_final columns: {X_final.shape[1]}")
print(f"2. X_final rows: {X_final.shape[0]}")
print(f"3. Memory: {X_final.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Check for object dtype columns
object_cols = X_final.select_dtypes(include=['object']).columns.tolist()
if object_cols:
    print(f"4. ⚠️ Object columns found: {len(object_cols)}")
    print(f"   Convert these to numeric or remove them")
    
    # Convert object columns to string then category
    for col in object_cols:
        X_final[col] = X_final[col].astype('category')
    print("   ✓ Converted to category dtype")

=== DIAGNOSTIC ===
1. X_final columns: 73
2. X_final rows: 25869
3. Memory: 9.23 MB


In [51]:
# 16. Save preprocessing objects for later use
import joblib

# Save scaler and imputer
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(imputer, '../models/imputer.pkl')

print("✓ Scaler saved to '../models/scaler.pkl'")
print("✓ Imputer saved to '../models/imputer.pkl'")
print(f"✓ Features used: {X_final.columns.tolist()[:10]}...")

✓ Scaler saved to '../models/scaler.pkl'
✓ Imputer saved to '../models/imputer.pkl'
✓ Features used: ['data_arc_in_days', 'observations_used', 'orbit_uncertainty', 'minimum_orbit_intersection', 'jupiter_tisserand_invariant', 'epoch_osculation', 'eccentricity', 'semi_major_axis', 'inclination', 'ascending_node_longitude']...


In [52]:
# 17. Preprocessing summary
print("=== PREPROCESSING SUMMARY ===")
print(f"1. Original shape: {df.shape}")
print(f"2. Features after cleaning: {X_final.shape[1]}")
print(f"3. Missing values handled: Yes (median/most_frequent)")
print(f"4. Scaling applied: RobustScaler")
print(f"5. Encoding applied: {'One-hot' if categorical_cols else 'None'}")
print(f"6. Train-test split: 80-20 with stratification")
print(f"7. Data saved for RF, DT, and Regression models")

=== PREPROCESSING SUMMARY ===
1. Original shape: (25869, 44)
2. Features after cleaning: 73
3. Missing values handled: Yes (median/most_frequent)
4. Scaling applied: RobustScaler
5. Encoding applied: One-hot
6. Train-test split: 80-20 with stratification
7. Data saved for RF, DT, and Regression models


In [53]:
# 18. Display final feature list
print("=== FINAL FEATURES FOR MODELING ===")
print(f"Total features: {X_final.shape[1]}")
print("\nSample features (first 15):")
for i, col in enumerate(X_final.columns[:15]):
    print(f"  {i+1}. {col}")
if X_final.shape[1] > 15:
    print(f"  ... and {X_final.shape[1] - 15} more features")

=== FINAL FEATURES FOR MODELING ===
Total features: 73

Sample features (first 15):
  1. data_arc_in_days
  2. observations_used
  3. orbit_uncertainty
  4. minimum_orbit_intersection
  5. jupiter_tisserand_invariant
  6. epoch_osculation
  7. eccentricity
  8. semi_major_axis
  9. inclination
  10. ascending_node_longitude
  11. orbital_period
  12. perihelion_distance
  13. perihelion_argument
  14. aphelion_distance
  15. perihelion_time
  ... and 58 more features


---
## Copyright Notice

**© 2026 StellarMind - EarthGuard Asteroid Defense AI. All Rights Reserved.**

This notebook and its contents are the intellectual property of StellarMind.  
No part of this code may be reproduced, distributed, or transmitted in any form without prior written permission from the authors.

**Authors**: Gouragopal Mohapatra & Arijit Kumar Mohanty  
**Project**: Asteroid Risk Prediction System

**Last Updated**: April 2026